In [162]:
import os
from pathlib import Path
import numpy as np
import pandas as pd

# Load Data 

In [163]:
data_path = Path("../data")

movie_path = f"{data_path}/refine_movies.csv"
rating_path = f"{data_path}/refine_ratings.csv"
tag_path = f"{data_path}/refine_tags.csv"
link_path = f"{data_path}/refine_links.csv"

In [164]:
movies_df = pd.read_csv(
    movie_path,
    usecols=['movieId', 'title','year','genres'],
    dtype={'title':str, 'year':str, 'movieId':int, 'genres':str})

ratings_df = pd.read_csv(
    rating_path,
    usecols=['userId','movieId','rating'],
    dtype={'userId':int, 'movieId':int, 'rating':float}
    )
tags_df = pd.read_csv(
    tag_path,
    usecols=['userId','movieId','tag'],
    dtype={'userId':int, 'movieId':int, 'tag':str})
links_df = pd.read_csv(link_path)

In [165]:
df_list = {
    "movies" : movies_df,
    "ratings" : ratings_df,
    "tags" : tags_df,
    "links_df" : links_df
}

# Data check

In [ ]:
# genres to list
import ast
movies_df['genres'] = movies_df['genres'].apply(lambda x : ast.literal_eval(x))

movies_df['genres']

0       [Adventure, Animation, Children, Comedy, Fantasy]
1                          [Adventure, Children, Fantasy]
2                                       [Comedy, Romance]
3                                [Comedy, Drama, Romance]
4                                                [Comedy]
                              ...                        
9734                 [Action, Animation, Comedy, Fantasy]
9735                         [Animation, Comedy, Fantasy]
9736                                              [Drama]
9737                                  [Action, Animation]
9738                                             [Comedy]
Name: genres, Length: 9739, dtype: object

In [167]:
ratings_df

,userId,movieId,rating
0,1,1,4.0
1,1,3,4.0
2,1,6,4.0
3,1,47,5.0
4,1,50,5.0
...,...,...,...
100831,610,166534,4.0
100832,610,168248,5.0
100833,610,168250,5.0
100834,610,168252,5.0


In [168]:
tags_df

,userId,movieId,tag
0,2,60756,['funny']
1,2,60756,"['Highly', 'quotable']"
2,2,60756,"['will', 'ferrell']"
3,2,89774,"['Boxing', 'story']"
4,2,89774,['MMA']
...,...,...,...
3678,606,7382,"['for', 'katie']"
3679,606,7936,['austere']
3680,610,3265,"['gun', 'fu']"
3681,610,3265,"['heroic', 'bloodshed']"


# Data Merge

In [219]:
movie_ratings_df = pd.merge(ratings_df, movies_df, on='movieId', how='left') # on : merge key
movie_ratings_df

,userId,movieId,rating,title,genres,year
0,1,1,4.0,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",(1995)
1,1,3,4.0,Grumpier Old Men (1995),"[Comedy, Romance]",(1995)
2,1,6,4.0,Heat (1995),"[Action, Crime, Thriller]",(1995)
3,1,47,5.0,Seven (a.k.a. Se7en) (1995),"[Mystery, Thriller]",(1995)
4,1,50,5.0,"Usual Suspects, The (1995)","[Crime, Mystery, Thriller]",(1995)
...,...,...,...,...,...,...
100831,610,166534,4.0,Split (2017),"[Drama, Horror, Thriller]",(2017)
100832,610,168248,5.0,John Wick: Chapter Two (2017),"[Action, Crime, Thriller]",(2017)
100833,610,168250,5.0,Get Out (2017),[Horror],(2017)
100834,610,168252,5.0,Logan (2017),"[Action, Sci-Fi]",(2017)


# zero ratings movie check

In [220]:
movie_ratings_df.isnull().sum()

userId     0
movieId    0
rating     0
title      0
genres     0
year       0
dtype: int64

In [221]:
no_ratings_movieId = movie_ratings_df.loc[movie_ratings_df.isnull().any(axis=1)]['movieId'].values
print("평가가 존재하지 않는 영화 ID 갯수: ",len(no_ratings_movieId))
no_ratings_movieId

평가가 존재하지 않는 영화 ID 갯수:  0


array([], dtype=int64)

In [222]:
movie_ratings_df[movie_ratings_df['movieId'].isin(no_ratings_movieId)]

,userId,movieId,rating,title,genres,year


In [226]:
# double check (ratings_df 에도 존재하지 않는지 확인)
ratings_df[ratings_df['movieId'].isin(no_ratings_movieId)]

,userId,movieId,rating


# user sim matrix

In [ ]:
item_user_matrix = movie_ratings_df.pivot_table(values='rating', index='title', columns='userId')

In [ ]:
item_user_matrix

userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
title,,,,,,,,,,,,,,,,,,,,,
'71 (2014),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0
'Hellboy': The Seeds of Creation (2004),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
'Round Midnight (1986),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
'Salem's Lot (2004),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
'Til There Was You (1997),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
eXistenZ (1999),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,5.0,NaN,NaN,NaN,NaN,4.5,NaN,NaN
xXx (2002),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.5,NaN,2.0
xXx: State of the Union (2005),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.5


In [243]:
print("평균 평점 갯수 :" ,round(item_user_matrix.apply(lambda x: x.count(), axis=1).mean()))
item_user_matrix.apply(lambda x: x.count(), axis=1)

평균 평점 갯수 : 10


title
'71 (2014)                                    1
'Hellboy': The Seeds of Creation (2004)       1
'Round Midnight (1986)                        2
'Salem's Lot (2004)                           1
'Til There Was You (1997)                     2
                                             ..
eXistenZ (1999)                              22
xXx (2002)                                   24
xXx: State of the Union (2005)                5
¡Three Amigos! (1986)                        26
À nous la liberté (Freedom for Us) (1931)     1
Length: 9721, dtype: int64

# fill NaN value in ratings df

## (1) Average ratings
- using movie_ratings_df / no_ratings_movieId

In [173]:
temp_df = movies_df[movies_df['movieId'].isin(no_ratings_movieId)].set_index("movieId")['genres']
movie_genre_dict = temp_df.to_dict()
movie_genre_dict

{1076: ['Drama', 'Horror', 'Thriller'],
 2939: ['Drama', 'Thriller'],
 3338: ['Documentary'],
 3456: ['Drama'],
 4194: ['Drama', 'Romance', 'War'],
 5721: ['Drama'],
 6668: ['Drama', 'Romance'],
 6849: ['Drama', 'Fantasy', 'Musical'],
 7020: ['Comedy', 'Drama', 'Romance'],
 7792: ['Thriller'],
 8765: ['Crime', 'Film-Noir', 'Thriller'],
 25855: ['Crime', 'Drama', 'Thriller'],
 26085: ['Adventure', 'Drama', 'Romance'],
 30892: ['Animation', 'Documentary'],
 32160: ['Comedy'],
 32371: ['Crime', 'Drama', 'Film-Noir'],
 34482: ['Drama'],
 85565: ['Comedy', 'Romance']}

In [185]:
ratings_df

,userId,movieId,rating
0,1,1,4.0
1,1,3,4.0
2,1,6,4.0
3,1,47,5.0
4,1,50,5.0
...,...,...,...
100831,610,166534,4.0
100832,610,168248,5.0
100833,610,168250,5.0
100834,610,168252,5.0


In [ ]:
# 유저별 평균 점수
user_grand_mean = ratings_df.groupby(by='userId')['rating'].mean()

In [ ]:
movies_df

,movieId,title,genres,year
0,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",(1995)
1,2,Jumanji (1995),"[Adventure, Children, Fantasy]",(1995)
2,3,Grumpier Old Men (1995),"[Comedy, Romance]",(1995)
3,4,Waiting to Exhale (1995),"[Comedy, Drama, Romance]",(1995)
4,5,Father of the Bride Part II (1995),[Comedy],(1995)
...,...,...,...,...
9734,193581,Black Butler: Book of the Atlantic (2017),"[Action, Animation, Comedy, Fantasy]",(2017)
9735,193583,No Game No Life: Zero (2017),"[Animation, Comedy, Fantasy]",(2017)
9736,193585,Flint (2017),[Drama],(2017)
9737,193587,Bungo Stray Dogs: Dead Apple (2018),"[Action, Animation]",(2018)
